# 29 – LangGraph State Machines

## Learning objectives

- Define graph state with `TypedDict` and wire nodes with `StateGraph`.
- Route with conditional edges and explicit `START` / `END`.
- Map the ReAct loop pattern (reason → act → observe) to a small graph.

## About this topic

LangGraph models workflows as state machines: each node reads state and returns a partial update. Typed state keeps keys explicit. Conditional edges let you branch on runtime values—ideal for tool loops, human approval gates, and retries.

In [ ]:
from pathlib import Path
import importlib.util

def _repo_root() -> Path:
    cwd = Path.cwd().resolve()
    return cwd if (cwd / "pyproject.toml").exists() else cwd.parent

path = _repo_root() / "examples/library/langgraph_workflows/basic_state_graph.py"
spec = importlib.util.spec_from_file_location("lg_basic", path)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
mod.demo_basic_state_graph()

In [ ]:
from pathlib import Path
import importlib.util

def _repo_root() -> Path:
    cwd = Path.cwd().resolve()
    return cwd if (cwd / "pyproject.toml").exists() else cwd.parent

path = _repo_root() / "examples/library/langgraph_workflows/react_agent.py"
spec = importlib.util.spec_from_file_location("lg_react", path)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
mod.demo_react_agent()

In [ ]:
from typing import Literal
from typing_extensions import TypedDict

class CounterState(TypedDict):
    n: int
    done: bool

def bump(s: CounterState) -> dict:
    return {"n": s["n"] + 1}

def finish(s: CounterState) -> dict:
    return {"done": True}

def route(s: CounterState) -> Literal["bump", "finish"]:
    return "finish" if s["n"] >= 2 else "bump"

try:
    from langgraph.graph import END, START, StateGraph
    g = StateGraph(CounterState)
    g.add_node("bump", bump)
    g.add_node("finish", finish)
    g.add_edge(START, "bump")
    g.add_conditional_edges("bump", route, {"bump": "bump", "finish": "finish"})
    g.add_edge("finish", END)
    app = g.compile()
    out = app.invoke({"n": 0, "done": False})
    print(out)
except ImportError:
    print("Install: pip install langgraph langchain-core")

## Exercises and next steps

1. Add a third node that logs the trace ID to state before `finish`.
2. Open `examples/library/langgraph_workflows/persistent_graph.py` and summarize how checkpointing changes your deployment assumptions.
3. Replace the mock `agent_node` in the ReAct example with a call to your preferred chat model and real tools.